# BYOC training — CatBoost credit model on SageMaker local mode

Build a **bring-your-own-container** (CatBoost), train it with SageMaker local mode (`instance_type='local'` — a real Docker training job), and store the model in oblako's **S3Proxy**. Mirrors `sagemaker_endpoints/ml_models` from the reference repo.

Prereqs: `oblako up s3proxy`, Docker, `pip install 'oblako[sagemaker,notebook]'`.

## Build the container + train

In [ ]:
import glob, os, tempfile
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")

import creditdata
from oblako.services import S3ProxyService, SageMakerService
from sagemaker.estimator import Estimator

IMAGE = "oblako-credit-risk:latest"
HP = {"iterations": "150", "depth": "6", "learning-rate": "0.1", "factor": "20", "offset": "600"}

sm = SageMakerService()
sm.build_image(path="container", tag=IMAGE)          # CatBoost BYOC (./container)

data_dir = tempfile.mkdtemp(prefix="credit-train-")
creditdata.write_training_csv(os.path.join(data_dir, "train.csv"), n=600)
out_dir = tempfile.mkdtemp(prefix="credit-out-")

estimator = Estimator(
    image_uri=IMAGE, role="arn:aws:iam::000000000000:role/dummy",
    instance_count=1, instance_type="local", sagemaker_session=sm.get_session(),
    output_path=f"file://{out_dir}", hyperparameters=HP,
)
estimator.fit({"train": f"file://{data_dir}"})

## Store the trained model in oblako's S3

In [ ]:
model_tar = glob.glob(os.path.join(out_dir, "**", "model.tar.gz"), recursive=True)[0]
s3 = S3ProxyService().get_client()
try:
    s3.create_bucket(Bucket="credit-risk-models")
except Exception:
    pass
s3.upload_file(model_tar, "credit-risk-models", "credit-risk/model.tar.gz")
print("stored: s3://credit-risk-models/credit-risk/model.tar.gz")
print("bucket:", [o["Key"] for o in s3.list_objects_v2(Bucket="credit-risk-models").get("Contents", [])])